To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth now supports [gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which automatically creates kernels!

[Vision RL](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) is now supported! Train Qwen2.5-VL, Gemma 3 etc. with GSPO or GRPO.

Introducing Unsloth [Standby for RL](https://docs.unsloth.ai/basics/memory-efficient-rl): GRPO is now faster, uses 30% less memory with 2x longer context.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [2]:
%%capture
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers
!uv pip install --no-deps trl==0.22.2

### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.11: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

In [5]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to(model.device)

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-10-01

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve equation: x^5 + 3x^4 - 10 = 3. So x^5 + 3x^4 -10 = 3 => x^5 + 3x^4 -13 = 0. Solve for x. It's quintic;


Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

In [6]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to(model.device)

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-10-01

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks: "Solve x^5 + 3x^4 - 10 = 3." Probably solve for x? Equation: x^5 + 3x^4 - 10 = 3. That simplifies to x^5 + 3x^4 - 13 =


Lastly we will test it using `reasoning_effort` to `high`

In [7]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to(model.device)

_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-10-01

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The prompt: "Solve x^5 + 3x^4 - 10 = 3." Solve the equation. So we set x^5 + 3x^4 - 10 = 3. That simplifies: x^5 + 3x^4 - 10 = 


<a name="Data"></a>
### Data Prep

The `HuggingFaceH4/Multilingual-Thinking` dataset will be utilized as our example. This dataset, available on Hugging Face, contains reasoning chain-of-thought examples derived from user questions that have been translated from English into four other languages. It is also the same dataset referenced in OpenAI's [cookbook](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers) for fine-tuning. The purpose of using this dataset is to enable the model to learn and develop reasoning capabilities in these four distinct languages.

In [8]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset

data_files = {
    "train": "/content/transactions_sft_train.jsonl",
    "validation": "/content/transactions_sft_val.jsonl"
}
dataset = load_dataset("json", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

To format our dataset, we will apply our version of the GPT OSS prompt

In [9]:
from datasets import load_dataset

# Already loaded earlier:
# dataset = load_dataset("json", data_files=data_files)

# Instead of standardize_sharegpt, define your own formatter:
def formatting_prompts_func(examples):
    inputs  = examples["input_text"]
    targets = examples["target_text"]
    return {
        "text": [f"### Input: {inp}\n### Output: {out}"
                 for inp, out in zip(inputs, targets)]
    }

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/42287 [00:00<?, ? examples/s]

Map:   0%|          | 0/4699 [00:00<?, ? examples/s]

Let's take a look at the dataset, and check what the 1st example shows

In [10]:
print("Train example:", dataset["train"][0])
print("Validation example:", dataset["validation"][0])


Train example: {'input_text': 'Transaction description: USPS', 'target_text': 'Payee: Usps Po 2804080611 Ballwin Mo | Category: Office expenses:Shipping & postage', 'text': '### Input: Transaction description: USPS\n### Output: Payee: Usps Po 2804080611 Ballwin Mo | Category: Office expenses:Shipping & postage'}
Validation example: {'input_text': 'Transaction description: Anisha Patel CPA', 'target_text': 'Payee: BUSINESS TO BUSINESS ACH PATEL & CO., INC SALE       230825                 REALM HEALTHCARE INC | Category: Legal & Professional Services', 'text': '### Input: Transaction description: Anisha Patel CPA\n### Output: Payee: BUSINESS TO BUSINESS ACH PATEL & CO., INC SALE       230825                 REALM HEALTHCARE INC | Category: Legal & Professional Services'}


In [11]:
from functools import partial

max_input_len  = 512   # adjust if your descriptions are longer
max_target_len = 128   # adjust if categories/payees are longer

def preprocess(batch, tokenizer):
    # Encode the inputs
    model_inputs = tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=max_input_len,
    )
    # Encode the labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target_text"],
            truncation=True,
            max_length=max_target_len,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing to both splits
tokenized_dataset = dataset.map(
    partial(preprocess, tokenizer=tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,  # drop input_text/target_text
)


Map:   0%|          | 0/42287 [00:00<?, ? examples/s]

Map:   0%|          | 0/4699 [00:00<?, ? examples/s]

What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [12]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset["train"],       # use the train split
    eval_dataset  = tokenized_dataset["validation"],  # add validation split
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Uncomment for full run
        max_steps = 30,  # quick test
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # change to "wandb" if you want logging
    ),
)


Unsloth: Switching to float32 training since model cannot work with float16


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes and lower loss as well!

In [14]:
"""
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(instruction_part = "<|start|>user<|message|>", response_part="<|start|>assistant<|channel|>final<|message|>")

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)
"""

'\nfrom unsloth.chat_templates import train_on_responses_only\n\ngpt_oss_kwargs = dict(instruction_part = "<|start|>user<|message|>", response_part="<|start|>assistant<|channel|>final<|message|>")\n\ntrainer = train_on_responses_only(\n    trainer,\n    **gpt_oss_kwargs,\n)\n'

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [15]:
# Pick any index you like
i = 100

# 1) Decode the model inputs (should look like your input_text prompt)
decoded_inputs = tokenizer.decode(
    trainer.train_dataset[i]["input_ids"],
    skip_special_tokens=True,
)
print("DECODED INPUT (should be input_text):\n", decoded_inputs)

# 2) Decode the labels (loss is computed only here).
# Replace ignore index (-100) with pad token for decoding only.
decoded_labels = tokenizer.decode(
    [tok if tok != -100 else tokenizer.pad_token_id
     for tok in trainer.train_dataset[i]["labels"]],
    skip_special_tokens=True,
)
print("\nDECODED LABELS (should be target_text):\n", decoded_labels)

# (Optional) Compare with the raw strings from the original split
print("\nRAW INPUT_TEXT:", dataset["train"][i]["input_text"])
print("RAW TARGET_TEXT:", dataset["train"][i]["target_text"])


DECODED INPUT (should be input_text):
 Transaction description: Lisa Merring

DECODED LABELS (should be target_text):
 Payee: 18x12SignatureBB | Category: Shopify Holding Account

RAW INPUT_TEXT: Transaction description: Lisa Merring
RAW TARGET_TEXT: Payee: 18x12SignatureBB | Category: Shopify Holding Account


Now let's print the masked out example - you should see only the answer is present:

In [ ]:
#tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [16]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
12.816 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [17]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998, 'pad_token_id': 200017}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 42,287 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,981,312 of 20,918,738,496 (0.02% trained)


Step,Training Loss
1,42.100300
2,40.417900
3,44.380400
4,72.121400
5,56.714800
6,46.871800
7,63.578100
8,58.171900
9,26.931800
10,39.020800


In [18]:
train_result = trainer.train()   # starts training
train_result                     # prints a summary


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 42,287 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,981,312 of 20,918,738,496 (0.02% trained)


Step,Training Loss
1,26.966300
2,26.253700
3,30.450000
4,47.829700
5,39.702200
6,32.641900
7,50.151000
8,45.029600
9,23.833800
10,31.079000


TrainOutput(global_step=30, training_loss=34.10794499715169, metrics={'train_runtime': 308.5692, 'train_samples_per_second': 0.389, 'train_steps_per_second': 0.097, 'total_flos': 1874498001960960.0, 'train_loss': 34.10794499715169, 'epoch': 0.0028377515548513726})

In [21]:
# Skip right to saving & inference, then run a smaller eval later:
trainer.save_model("outputs-final")
tokenizer.save_pretrained("outputs-final")



('outputs-final/tokenizer_config.json',
 'outputs-final/special_tokens_map.json',
 'outputs-final/chat_template.jinja',
 'outputs-final/tokenizer.json')

In [19]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

437.0187 seconds used for training.
7.28 minutes used for training.
Peak reserved memory = 12.816 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 86.941 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [24]:
"""
messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))
"""

'\nmessages = [\n    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},\n    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},\n]\ninputs = tokenizer.apply_chat_template(\n    messages,\n    add_generation_prompt = True,\n    return_tensors = "pt",\n    return_dict = True,\n    reasoning_effort = "medium",\n).to(model.device)\nfrom transformers import TextStreamer\n_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))\n'

In [42]:
import re, torch, difflib
from transformers import StoppingCriteria, StoppingCriteriaList, LogitsProcessor, LogitsProcessorList

# ---------- helpers ----------
def _decode_completion(outputs, inputs, tokenizer):
    gen_only = outputs[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_only, skip_special_tokens=True)

def _clean_field(s: str) -> str:
    s = s.strip()
    s = re.sub(r"^[\d\.\-\•\*]+\s*", "", s)     # strip bullets/numbering at start
    s = re.sub(r"\s*\|.*$", "", s)              # cut anything after a pipe
    return s.strip()

def _mostly_digits(s: str, min_len=2):
    s2 = re.sub(r"\s+","", s)
    return len(s2) >= min_len and sum(ch.isdigit() for ch in s2) / max(1,len(s2)) > 0.6

def heuristic_payee_from_desc(description: str) -> str:
    d = re.sub(r"\$?\d[\d,\.\/:-]*", " ", description)
    candidates = re.findall(r"\b(?:[A-Z][A-Za-z&.\'-]{2,}(?:\s+[A-Z][A-Za-z&.\'-]{2,})*)\b", d)
    if not candidates:
        words = re.findall(r"[A-Za-z][A-Za-z&.\'-]+", description)
        return " ".join(words[:3]).strip()
    return max(candidates, key=len)[:64].strip()

def normalize_category(cat: str, allowed):
    cat = cat.strip()
    if not allowed:
        return cat
    if cat in allowed:
        return cat
    m = difflib.get_close_matches(cat, allowed, n=1, cutoff=0.72)
    return m[0] if m else ""   # empty => “no suitable category”

# ---------- stopping criterion with minimum generated tokens ----------
class StopOnSequenceMinLen(StoppingCriteria):
    def __init__(self, tokenizer, stop_str, min_gen_tokens=0, prompt_len=None):
        self.stop_ids = tokenizer(stop_str, add_special_tokens=False)["input_ids"]
        self.k = len(self.stop_ids)
        self.min_gen_tokens = min_gen_tokens
        self.prompt_len = prompt_len
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        if self.prompt_len is None or input_ids.shape[1] < self.k:
            return False
        gen_tokens = input_ids.shape[1] - self.prompt_len
        if gen_tokens < self.min_gen_tokens:
            return False
        tail = input_ids[0, -self.k:].tolist()
        return tail == self.stop_ids

# ---------- discourage leading digits for payee ----------
class BanLeadingDigits(LogitsProcessor):
    def __init__(self, tokenizer, start_pos_tracker):
        self.tok = tokenizer
        self.start_pos_tracker = start_pos_tracker
        enc = self.tok(list("0123456789"), add_special_tokens=False, is_split_into_words=True)
        digit_ids = set()
        for seq in enc["input_ids"]:
            for tid in (seq if isinstance(seq, list) else [seq]):
                digit_ids.add(int(tid))
        self.digit_ids = digit_ids
    def __call__(self, input_ids, scores):
        gen_len = input_ids.shape[1] - self.start_pos_tracker["prompt_len"]
        if gen_len <= 2:
            for did in self.digit_ids:
                scores[0, did] = -1e9
        return scores

# ---------- prompts ----------
SYSTEM_FMT = (
    "You are an extraction model. Given a transaction description, output exactly one line:\n"
    "Payee: <name> | Category: <category>\n"
    "Rules:\n"
    " - Payee MUST NOT be empty. Extract a concise vendor/merchant name from the description.\n"
    " - Category MUST be chosen from the provided list.\n"
    " - Do not add numbering, bullets, or extra text.\n"
)

FEWSHOTS = """Examples:
Transaction description: Starbucks Coffee #123 Seattle
Answer: Payee: Starbucks | Category: Meals & Entertainment

Transaction description: Uber trip NYC 08/11
Answer: Payee: Uber | Category: Travel

Transaction description: Amazon Web Services – Cloud Hosting
Answer: Payee: Amazon Web Services | Category: Cloud Hosting
"""

def predict_payee_category_constrained(description,
                                       max_new_tokens_payee=32,
                                       max_new_tokens_cat=32,
                                       payee_min_tokens=4):
    # ---- PAYEE step (as before) ----
    prompt1 = (
        f"{SYSTEM_FMT}\n\n{FEWSHOTS}\n"
        f"Transaction description: {description}\n"
        f"Answer: Payee: "
    )
    inputs1 = tokenizer(prompt1, return_tensors="pt").to(model.device)
    start_tracker = {"prompt_len": inputs1["input_ids"].shape[1]}
    stop_bar = StopOnSequenceMinLen(tokenizer, " |", min_gen_tokens=payee_min_tokens, prompt_len=start_tracker["prompt_len"])
    stop_nl  = StopOnSequenceMinLen(tokenizer, "\n",  min_gen_tokens=payee_min_tokens, prompt_len=start_tracker["prompt_len"])

    with torch.no_grad():
        out1 = model.generate(
            **inputs1,
            max_new_tokens=max_new_tokens_payee,
            do_sample=False, temperature=0.0, top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=LogitsProcessorList([BanLeadingDigits(tokenizer, start_tracker)]),
            stopping_criteria=StoppingCriteriaList([stop_bar, stop_nl]),
            eos_token_id=[tokenizer.eos_token_id, tokenizer("\n", add_special_tokens=False)["input_ids"][0]],
        )
    payee_raw = _decode_completion(out1, inputs1, tokenizer)
    payee = _clean_field(payee_raw)
    if (not payee) or _mostly_digits(payee):
        payee = heuristic_payee_from_desc(description)

    # ---- CATEGORY step (constrained to whitelist) ----
    # Present the valid options and force choosing exactly one.
    cat_list = allowed_categories if 'allowed_categories' in globals() else []
    cat_instruction = ""
    if cat_list:
        # Keep the list compact; models do fine with comma-separated options.
        cat_instruction = "Valid categories:\n- " + "\n- ".join(cat_list[:150])  # limit to first 150 if very large

    prompt2 = (
        f"{SYSTEM_FMT}\n{cat_instruction}\n\n"
        f"Transaction description: {description}\n"
        f"Answer: Payee: {payee} | Category: "
    )
    inputs2 = tokenizer(prompt2, return_tensors="pt").to(model.device)
    stop_nl2 = StopOnSequenceMinLen(tokenizer, "\n", min_gen_tokens=1, prompt_len=inputs2["input_ids"].shape[1])

    with torch.no_grad():
        out2 = model.generate(
            **inputs2,
            max_new_tokens=max_new_tokens_cat,
            do_sample=False, temperature=0.0, top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([stop_nl2]),
            eos_token_id=[tokenizer.eos_token_id, tokenizer("\n", add_special_tokens=False)["input_ids"][0]],
        )
    category_raw = _decode_completion(out2, inputs2, tokenizer)
    category = _clean_field(category_raw)
    category = normalize_category(category, cat_list)

    return f"Payee: {payee} | Category: {category}", (payee, category)

# ---- Try it ----
raw, (p, c) = predict_payee_category_constrained("Amazon Web Services – Cloud Hosting")
print("RAW:", raw)
print("PARSED:", p, "|", c)


OutOfMemoryError: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 6.12 MiB is free. Process 2285 has 14.73 GiB memory in use. Of the allocated memory 14.43 GiB is allocated by PyTorch, with 24.00 MiB allocated in private pools (e.g., CUDA Graphs), and 51.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [39]:
print("tokenizer in session:", 'tokenizer' in globals())
print("model in session    :", 'model' in globals())


tokenizer in session: True
model in session    : True


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** Currently finetunes can only be loaded via Unsloth in the meantime - we're working on vLLM and GGUF exporting!

In [ ]:
model.save_pretrained("finetuned_model")
# model.push_to_hub("hf_username/finetuned_model", token = "hf_...") # Save to HF

To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "finetuned_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 1024,
        dtype = None,
        load_in_4bit = True,
    )

messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to(model.device)
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are a helpful assistant that can solve mathematical problems.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the equation for x. The equation: x^5 + 3x^4 - 10 = 3. So bring 3 to left side: x^5 + 3x^4 -10 -3 = 0 → x^5 + 3x^


### Saving to float16 for VLLM or mxfp4

We also support saving to `float16` or `mxfp4` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method="mxfp4")
if False: model.push_to_hub_merged("repo_id/repo_name", tokenizer, token="hf...", save_method="mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method="merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("hf/gpt-oss-finetune", tokenizer, save_method = "merged_16bit", token = "")

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
